In [4]:
from pathlib import Path
import pandas as pd

# Base folder
BASE_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\raw\tabular\2021 Census GCP All Geographies for VIC")

# Output folder
OUTPUT_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_combined")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GEOGRAPHIES = ["SA2", "SAL", "POA"]


def load_and_merge_geography(geo: str) -> pd.DataFrame:
    geo_dir = BASE_DIR / geo / "VIC"
    files = sorted(geo_dir.glob("*.csv"))

    if not files:
        raise FileNotFoundError(f"No CSV files found in {geo_dir}")

    merged = None
    key_col = None

    for file in files:
        df = pd.read_csv(file, low_memory=False)

        current_key = df.columns[0]
        if key_col is None:
            key_col = current_key
        elif current_key != key_col:
            df = df.rename(columns={current_key: key_col})

        df = df.drop_duplicates(subset=[key_col])

        if merged is None:
            merged = df.copy()
            print(f"Loaded {file.name}: {df.shape[0]} rows, {df.shape[1]} cols")
            continue

        # Drop overlapping non-key columns before merge
        overlap_cols = [c for c in df.columns if c != key_col and c in merged.columns]
        if overlap_cols:
            print(f"Dropping {len(overlap_cols)} overlapping columns from {file.name}")
            df = df.drop(columns=overlap_cols)

        merged = pd.merge(merged, df, on=key_col, how="outer")
        print(f"Loaded {file.name}: {df.shape[0]} rows, {df.shape[1]} cols -> merged shape {merged.shape}")

    merged = merged.loc[:, ~merged.columns.duplicated()]
    print(f"\nFinal merged {geo}: {merged.shape[0]} rows, {merged.shape[1]} cols")
    return merged


for geo in GEOGRAPHIES:
    print(f"\n{'='*60}")
    print(f"Processing {geo}")
    print(f"{'='*60}")

    merged_df = load_and_merge_geography(geo)

    output_file = OUTPUT_DIR / f"census_{geo.lower()}_all_topics.csv"
    merged_df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")



Processing SA2
Loaded 2021Census_G01_VIC_SA2.csv: 524 rows, 109 cols
Loaded 2021Census_G02_VIC_SA2.csv: 524 rows, 9 cols -> merged shape (524, 117)
Loaded 2021Census_G03_VIC_SA2.csv: 524 rows, 141 cols -> merged shape (524, 257)
Loaded 2021Census_G04A_VIC_SA2.csv: 524 rows, 201 cols -> merged shape (524, 457)
Loaded 2021Census_G04B_VIC_SA2.csv: 524 rows, 107 cols -> merged shape (524, 563)
Loaded 2021Census_G05_VIC_SA2.csv: 524 rows, 181 cols -> merged shape (524, 743)
Loaded 2021Census_G06_VIC_SA2.csv: 524 rows, 121 cols -> merged shape (524, 863)
Loaded 2021Census_G07_VIC_SA2.csv: 524 rows, 181 cols -> merged shape (524, 1043)
Loaded 2021Census_G08_VIC_SA2.csv: 524 rows, 199 cols -> merged shape (524, 1241)
Loaded 2021Census_G09A_VIC_SA2.csv: 524 rows, 201 cols -> merged shape (524, 1441)
Loaded 2021Census_G09B_VIC_SA2.csv: 524 rows, 201 cols -> merged shape (524, 1641)
Dropping 1 overlapping columns from 2021Census_G09C_VIC_SA2.csv
Loaded 2021Census_G09C_VIC_SA2.csv: 524 rows, 200 

In [5]:
from collections import Counter

geo = "SA2"
geo_dir = BASE_DIR / geo / "VIC"
counter = Counter()

for file in sorted(geo_dir.glob("*.csv")):
    df = pd.read_csv(file, nrows=0)
    cols = list(df.columns[1:])  # skip key column
    counter.update(cols)

dupes = {k: v for k, v in counter.items() if v > 1}
print(f"Duplicate non-key columns: {len(dupes)}")
print(list(dupes.items())[:20])


Duplicate non-key columns: 415
[('Total_Total', 5), ('Tot_M', 6), ('Tot_F', 6), ('Tot_P', 6), ('M_15_19_yr_Tot', 2), ('M_20_24_yr_Tot', 2), ('M_25_34_yr_Tot', 2), ('M_35_44_yr_Tot', 2), ('M_45_54_yr_Tot', 2), ('M_55_64_yr_Tot', 2), ('M_65_74_yr_Tot', 2), ('M_75_84_yr_Tot', 2), ('M_85ov_Tot', 2), ('M_Tot_Tot', 19), ('F_15_19_yr_Tot', 2), ('F_20_24_yr_Tot', 2), ('F_25_34_yr_Tot', 2), ('F_35_44_yr_Tot', 2), ('F_45_54_yr_Tot', 2), ('F_55_64_yr_Tot', 2)]


In [6]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\raw\tabular\2021 Census GCP All Geographies for VIC")
OUTPUT_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_combined_prefixed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GEOGRAPHIES = ["SA2", "SAL", "POA"]


def extract_topic_code(filename: str) -> str:
    # Example: 2021Census_G01_VIC_SA2.csv -> G01
    parts = filename.replace(".csv", "").split("_")
    return parts[1]


def load_and_merge_geography_prefixed(geo: str) -> pd.DataFrame:
    geo_dir = BASE_DIR / geo / "VIC"
    files = sorted(geo_dir.glob("*.csv"))

    if not files:
        raise FileNotFoundError(f"No CSV files found in {geo_dir}")

    merged = None
    key_col = None

    for file in files:
        df = pd.read_csv(file, low_memory=False)
        topic_code = extract_topic_code(file.name)

        current_key = df.columns[0]

        if key_col is None:
            key_col = current_key
        elif current_key != key_col:
            df = df.rename(columns={current_key: key_col})

        df = df.drop_duplicates(subset=[key_col])

        rename_map = {}
        for col in df.columns:
            if col != key_col:
                rename_map[col] = f"{topic_code}_{col}"

        df = df.rename(columns=rename_map)

        if merged is None:
            merged = df.copy()
        else:
            merged = pd.merge(merged, df, on=key_col, how="outer")

        print(f"Loaded {file.name}: {df.shape[0]} rows, {df.shape[1]} cols")

    print(f"\nFinal merged {geo}: {merged.shape[0]} rows, {merged.shape[1]} cols")
    return merged


for geo in GEOGRAPHIES:
    print(f"\n{'='*60}")
    print(f"Processing {geo}")
    print(f"{'='*60}")

    merged_df = load_and_merge_geography_prefixed(geo)

    output_file = OUTPUT_DIR / f"census_{geo.lower()}_all_topics_prefixed.csv"
    merged_df.to_csv(output_file, index=False)

    print(f"Saved: {output_file}")



Processing SA2
Loaded 2021Census_G01_VIC_SA2.csv: 524 rows, 109 cols
Loaded 2021Census_G02_VIC_SA2.csv: 524 rows, 9 cols
Loaded 2021Census_G03_VIC_SA2.csv: 524 rows, 141 cols
Loaded 2021Census_G04A_VIC_SA2.csv: 524 rows, 201 cols
Loaded 2021Census_G04B_VIC_SA2.csv: 524 rows, 107 cols
Loaded 2021Census_G05_VIC_SA2.csv: 524 rows, 181 cols
Loaded 2021Census_G06_VIC_SA2.csv: 524 rows, 121 cols
Loaded 2021Census_G07_VIC_SA2.csv: 524 rows, 181 cols
Loaded 2021Census_G08_VIC_SA2.csv: 524 rows, 199 cols
Loaded 2021Census_G09A_VIC_SA2.csv: 524 rows, 201 cols
Loaded 2021Census_G09B_VIC_SA2.csv: 524 rows, 201 cols
Loaded 2021Census_G09C_VIC_SA2.csv: 524 rows, 201 cols
Loaded 2021Census_G09D_VIC_SA2.csv: 524 rows, 201 cols
Loaded 2021Census_G09E_VIC_SA2.csv: 524 rows, 201 cols
Loaded 2021Census_G09F_VIC_SA2.csv: 524 rows, 201 cols
Loaded 2021Census_G09G_VIC_SA2.csv: 524 rows, 201 cols
Loaded 2021Census_G09H_VIC_SA2.csv: 524 rows, 191 cols
Loaded 2021Census_G10A_VIC_SA2.csv: 524 rows, 201 cols
Loa

In [7]:
for geo in ["sa2", "sal", "poa"]:
    path = OUTPUT_DIR / f"census_{geo}_all_topics_prefixed.csv"
    df = pd.read_csv(path, nrows=5, low_memory=False)
    print(path.name, df.shape)
    print(df.columns[:10].tolist())
    print()


census_sa2_all_topics_prefixed.csv (5, 16985)
['SA2_CODE_2021', 'G01_Tot_P_M', 'G01_Tot_P_F', 'G01_Tot_P_P', 'G01_Age_0_4_yr_M', 'G01_Age_0_4_yr_F', 'G01_Age_0_4_yr_P', 'G01_Age_5_14_yr_M', 'G01_Age_5_14_yr_F', 'G01_Age_5_14_yr_P']

census_sal_all_topics_prefixed.csv (5, 16985)
['SAL_CODE_2021', 'G01_Tot_P_M', 'G01_Tot_P_F', 'G01_Tot_P_P', 'G01_Age_0_4_yr_M', 'G01_Age_0_4_yr_F', 'G01_Age_0_4_yr_P', 'G01_Age_5_14_yr_M', 'G01_Age_5_14_yr_F', 'G01_Age_5_14_yr_P']

census_poa_all_topics_prefixed.csv (5, 16985)
['POA_CODE_2021', 'G01_Tot_P_M', 'G01_Tot_P_F', 'G01_Tot_P_P', 'G01_Age_0_4_yr_M', 'G01_Age_0_4_yr_F', 'G01_Age_0_4_yr_P', 'G01_Age_5_14_yr_M', 'G01_Age_5_14_yr_F', 'G01_Age_5_14_yr_P']



In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

INPUT_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_combined_prefixed")
OUTPUT_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "sa2": INPUT_DIR / "census_sa2_all_topics_prefixed.csv",
    "sal": INPUT_DIR / "census_sal_all_topics_prefixed.csv",
    "poa": INPUT_DIR / "census_poa_all_topics_prefixed.csv",
}

MISSING_MARKERS = ["..", "...", "-", "NA", "N/A", "na", "n/a", "null", "NULL", ""]


def clean_census_file(name, path):
    print(f"\n{'='*70}")
    print(f"CLEANING {name.upper()}")
    print(f"{'='*70}")

    # Read as strings first so we can clean consistently
    df = pd.read_csv(path, dtype=str, keep_default_na=False, low_memory=False)

    key_col = df.columns[0]
    print(f"Original shape: {df.shape}")
    print(f"Key column: {key_col}")

    # Strip whitespace from all string cells
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

    # Replace common Census missing markers
    df = df.replace(MISSING_MARKERS, np.nan)
    df = df.replace(r"^\s*$", np.nan, regex=True)

    # Drop rows with missing key
    before = len(df)
    df = df[df[key_col].notna()].copy()
    print(f"Dropped rows with missing key: {before - len(df)}")

    # Drop duplicate keys, keep first
    before = len(df)
    df = df.drop_duplicates(subset=[key_col], keep="first").copy()
    print(f"Dropped duplicate keys: {before - len(df)}")

    # Drop fully empty columns
    all_null_cols = [c for c in df.columns if df[c].isna().all()]
    df = df.drop(columns=all_null_cols)
    print(f"Dropped all-null columns: {len(all_null_cols)}")

    # Drop fully empty rows except key
    non_key_cols = [c for c in df.columns if c != key_col]
    before = len(df)
    if non_key_cols:
        df = df.dropna(subset=non_key_cols, how="all").copy()
    print(f"Dropped fully empty rows: {before - len(df)}")

    # Convert columns to numeric where possible
    numeric_converted = 0
    for col in non_key_cols:
        if col not in df.columns:
            continue

        original_non_null = df[col].notna().sum()
        if original_non_null == 0:
            continue

        converted = pd.to_numeric(df[col], errors="coerce")
        converted_non_null = converted.notna().sum()

        # Convert only if >= 95% of non-null values parse as numeric
        if original_non_null > 0 and (converted_non_null / original_non_null) >= 0.95:
            df[col] = converted
            numeric_converted += 1

    print(f"Converted mostly-numeric columns: {numeric_converted}")

    # Optional: sort by key
    df = df.sort_values(by=key_col).reset_index(drop=True)

    output_path = OUTPUT_DIR / f"census_{name}_all_topics_cleaned.csv"
    df.to_csv(output_path, index=False)

    print(f"Cleaned shape: {df.shape}")
    print(f"Saved: {output_path}")

    return df


cleaned = {}
for name, path in FILES.items():
    cleaned[name] = clean_census_file(name, path)



CLEANING SA2
Original shape: (524, 16985)
Key column: SA2_CODE_2021
Dropped rows with missing key: 0
Dropped duplicate keys: 0
Dropped all-null columns: 228
Dropped fully empty rows: 0
Converted mostly-numeric columns: 16756
Cleaned shape: (524, 16757)
Saved: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_cleaned\census_sa2_all_topics_cleaned.csv

CLEANING SAL
Original shape: (2946, 16985)
Key column: SAL_CODE_2021
Dropped rows with missing key: 0
Dropped duplicate keys: 0
Dropped all-null columns: 228
Dropped fully empty rows: 0
Converted mostly-numeric columns: 16756
Cleaned shape: (2946, 16757)
Saved: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_cleaned\census_sal_all_topics_cleaned.csv

CLEANING POA
Original shape: (694, 16985)
Key column: POA_CODE_2021
Dropped rows with missing key: 0
Dropped duplicate keys: 0
Dropped all-null columns: 228
Dropped fully empty rows: 0
Converted mostly-numeric columns: 16756
Cleaned shape: (694, 16757

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

INPUT_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_cleaned")
OUTPUT_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_profiles")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "sa2": INPUT_DIR / "census_sa2_all_topics_cleaned.csv",
    "sal": INPUT_DIR / "census_sal_all_topics_cleaned.csv",
    "poa": INPUT_DIR / "census_poa_all_topics_cleaned.csv",
}


def pct(numerator, denominator):
    return np.where(
        (denominator.notna()) & (denominator != 0),
        (numerator / denominator * 100).round(2),
        np.nan
    )


def build_profile(path: Path, geo_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    key_col = df.columns[0]

    out = pd.DataFrame()
    out[key_col] = df[key_col]

    # Core counts / medians
    out["total_population"] = df["G01_Tot_P_P"]
    out["median_age"] = df["G02_Median_age_persons"]
    out["median_household_income_weekly"] = df["G02_Median_tot_hhd_inc_weekly"]
    out["median_personal_income_weekly"] = df["G02_Median_tot_prsnl_inc_weekly"]
    out["median_family_income_weekly"] = df["G02_Median_tot_fam_inc_weekly"]
    out["median_rent_weekly"] = df["G02_Median_rent_weekly"]
    out["median_mortgage_monthly"] = df["G02_Median_mortgage_repay_monthly"]
    out["average_household_size"] = df["G02_Average_household_size"]

    # Age profile
    out["age_0_14_count"] = df["G01_Age_0_4_yr_P"] + df["G01_Age_5_14_yr_P"]
    out["age_15_24_count"] = df["G01_Age_15_19_yr_P"] + df["G01_Age_20_24_yr_P"]
    out["age_25_34_count"] = df["G01_Age_25_34_yr_P"]
    out["age_65_plus_count"] = (
        df["G01_Age_65_74_yr_P"] +
        df["G01_Age_75_84_yr_P"] +
        df["G01_Age_85ov_P"]
    )

    out["age_0_14_pct"] = pct(out["age_0_14_count"], out["total_population"])
    out["age_15_24_pct"] = pct(out["age_15_24_count"], out["total_population"])
    out["age_25_34_pct"] = pct(out["age_25_34_count"], out["total_population"])
    out["age_65_plus_pct"] = pct(out["age_65_plus_count"], out["total_population"])

    # Diversity
    out["born_overseas_count"] = df["G01_Birthplace_Elsewhere_P"]
    out["language_other_than_english_count"] = df["G01_Lang_used_home_Oth_Lang_P"]

    out["born_overseas_pct"] = pct(out["born_overseas_count"], out["total_population"])
    out["language_other_than_english_pct"] = pct(
        out["language_other_than_english_count"], out["total_population"]
    )

    # Need for assistance
    out["need_for_assistance_count"] = df["G18_P_Tot_Need_for_assistance"]
    out["need_for_assistance_pct"] = pct(
        out["need_for_assistance_count"], df["G18_P_Tot_Tot"]
    )

    # Family / household profile
    out["couple_family_with_children_count"] = df["G29_CF_ChU15_a_Total_P"]
    out["one_parent_family_count"] = df["G29_OPF_Total_P"]
    out["other_family_count"] = df["G29_Other_family_P"]
    out["total_families"] = df["G32_Tot_Tot"]

    out["couple_family_with_children_pct"] = pct(
        out["couple_family_with_children_count"], out["total_families"]
    )
    out["one_parent_family_pct"] = pct(
        out["one_parent_family_count"], out["total_families"]
    )

    # Household type
    out["family_households_count"] = df["G33_Tot_Family_households"]
    out["non_family_households_count"] = df["G33_Tot_Non_family_households"]
    out["total_households"] = df["G33_Tot_Tot"]

    out["family_households_pct"] = pct(
        out["family_households_count"], out["total_households"]
    )
    out["non_family_households_pct"] = pct(
        out["non_family_households_count"], out["total_households"]
    )

    # Lone person / group households
    out["lone_person_households_count"] = df["G42_Tot_Lone_P_H"]
    out["group_households_count"] = df["G42_Tot_Group_H"]

    out["lone_person_households_pct"] = pct(
        out["lone_person_households_count"], df["G42_Tot_Tot"]
    )
    out["group_households_pct"] = pct(
        out["group_households_count"], df["G42_Tot_Tot"]
    )

    # Cars
    out["no_car_households_count"] = df["G34_Num_MVs_per_dweling_0_MVs"]
    out["one_car_households_count"] = df["G34_Num_MVs_per_dweling_1_MVs"]
    out["two_car_households_count"] = df["G34_Num_MVs_per_dweling_2_MVs"]
    out["three_plus_car_households_count"] = (
        df["G34_Num_MVs_per_dweling_3_MVs"] +
        df["G34_Num_MVs_per_dweling_4mo_MVs"]
    )
    out["car_households_total"] = df["G34_Total_dwelings"]

    out["no_car_households_pct"] = pct(
        out["no_car_households_count"], out["car_households_total"]
    )
    out["two_plus_car_households_pct"] = pct(
        out["two_car_households_count"] + out["three_plus_car_households_count"],
        out["car_households_total"]
    )

    # Tenure
    out["owned_outright_count"] = df["G37_O_OR_Total"]
    out["owned_with_mortgage_count"] = df["G37_O_MTG_Total"]
    out["rented_count"] = df["G37_R_Tot_Total"]
    out["tenure_total"] = df["G37_Total_Total"]

    out["owned_outright_pct"] = pct(
        out["owned_outright_count"], out["tenure_total"]
    )
    out["owned_with_mortgage_pct"] = pct(
        out["owned_with_mortgage_count"], out["tenure_total"]
    )
    out["renters_pct"] = pct(
        out["rented_count"], out["tenure_total"]
    )
    out["owner_occupied_pct"] = pct(
        out["owned_outright_count"] + out["owned_with_mortgage_count"],
        out["tenure_total"]
    )

    # Travel to work
    out["train_to_work_count"] = df["G62_One_method_Train_P"]
    out["bus_to_work_count"] = df["G62_One_method_Bus_P"]
    out["tram_to_work_count"] = df["G62_One_met_Tram_or_lt_rail_P"]
    out["car_driver_to_work_count"] = df["G62_One_method_Car_as_driver_P"]
    out["walk_to_work_count"] = df["G62_One_method_Walked_only_P"]
    out["worked_from_home_count"] = df["G62_Worked_home_P"]
    out["travel_to_work_total"] = df["G62_Tot_P"]

    out["public_transport_to_work_pct"] = pct(
        out["train_to_work_count"] + out["bus_to_work_count"] + out["tram_to_work_count"],
        out["travel_to_work_total"]
    )
    out["car_to_work_pct"] = pct(
        out["car_driver_to_work_count"], out["travel_to_work_total"]
    )
    out["walk_to_work_pct"] = pct(
        out["walk_to_work_count"], out["travel_to_work_total"]
    )
    out["worked_from_home_pct"] = pct(
        out["worked_from_home_count"], out["travel_to_work_total"]
    )

    # Optional affordability pressure proxy
    out["rent_to_income_ratio"] = np.where(
        (out["median_household_income_weekly"].notna()) &
        (out["median_household_income_weekly"] != 0),
        (out["median_rent_weekly"] / out["median_household_income_weekly"]).round(4),
        np.nan
    )

    # Clean infinities just in case
    out = out.replace([np.inf, -np.inf], np.nan)

    # Save
    output_path = OUTPUT_DIR / f"census_{geo_name}_profile.csv"
    out.to_csv(output_path, index=False)

    print(f"{geo_name.upper()} -> {out.shape} saved to {output_path}")
    return out


profiles = {}
for geo_name, path in FILES.items():
    profiles[geo_name] = build_profile(path, geo_name)


SA2 -> (524, 65) saved to C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_profiles\census_sa2_profile.csv
SAL -> (2946, 65) saved to C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_profiles\census_sal_profile.csv
POA -> (694, 65) saved to C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\census_profiles\census_poa_profile.csv
